# ⚙️ Notebook 2: Feature Engineering
## Climate-Disease Africa
---
**Author:** Emmanuel Yaw Afram (Prestige) | eyafram7@gmail.com

**Objective:** Create features that capture how climate drives disease with time delays.

## 1. Setup & Load

In [ ]:
import sys,warnings
warnings.filterwarnings('ignore')
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
sys.path.insert(0,'../src')
from data_loader import DiseaseDataLoader
from preprocessing import DiseasePreprocessor

df = DiseaseDataLoader().load_all_data()
df['date'] = pd.to_datetime(df['date'])
print(f'Loaded: {df.shape}')

## 2. Why Lag Features?

Diseases don't respond to climate instantly:
- Mosquitoes need **2-4 weeks** to breed after rainfall
- Cholera spreads **days-weeks** after flooding contaminates water
- We capture this with **lag features**: climate from 1, 2, 3 months ago

In [ ]:
# Demonstrate lag: show Malaria vs Precipitation 1 month ago
loc = df[df['country']=='Nigeria'].sort_values('date').copy()
loc['precip_lag1'] = loc['precipitation'].shift(1)

fig,axes = plt.subplots(1,2,figsize=(13,4))
axes[0].scatter(loc['precipitation'],loc['malaria_cases'],alpha=0.3,s=10,color='#3498DB')
axes[0].set_title('Malaria vs Current Rainfall')
axes[0].set_xlabel('Precipitation (mm)'); axes[0].set_ylabel('Malaria Cases')
r0 = loc[['precipitation','malaria_cases']].corr().iloc[0,1]
axes[0].text(0.05,0.92,f'r = {r0:.3f}',transform=axes[0].transAxes,
             bbox=dict(boxstyle='round',facecolor='lightyellow'))

valid = loc.dropna(subset=['precip_lag1'])
axes[1].scatter(valid['precip_lag1'],valid['malaria_cases'],alpha=0.3,s=10,color='#E74C3C')
axes[1].set_title('Malaria vs Rainfall 1 Month Ago (LAG=1)')
axes[1].set_xlabel('Precipitation lag-1 (mm)'); axes[1].set_ylabel('Malaria Cases')
r1 = valid[['precip_lag1','malaria_cases']].corr().iloc[0,1]
axes[1].text(0.05,0.92,f'r = {r1:.3f} (STRONGER)',transform=axes[1].transAxes,
             bbox=dict(boxstyle='round',facecolor='lightgreen'))
plt.suptitle('Lag Features Improve Correlation',fontweight='bold')
plt.tight_layout(); plt.show()
print(f'No lag: r={r0:.3f} | With 1-month lag: r={r1:.3f}')

## 3. Run Full Feature Engineering

In [ ]:
prep = DiseasePreprocessor()
df_clean = prep.clean_data(df)
df_features = prep.engineer_features(df_clean)

new_cols = [c for c in df_features.columns if c not in df_clean.columns]
print(f'Original columns : {df_clean.shape[1]}')
print(f'After engineering: {df_features.shape[1]}')
print(f'New features added: {len(new_cols)}')
for c in new_cols[:15]:
    print(f'  + {c}')

## 4. Heat-Humidity Stress Index

In [ ]:
# heat_humid_stress = temperature × (humidity/100)
# High values = ideal conditions for mosquito breeding
print('Heat-Humidity Stress Statistics:')
print(df_features['heat_humid_stress'].describe().round(2))

fig,ax = plt.subplots(figsize=(10,4))
ax.scatter(df_features['heat_humid_stress'].sample(3000,random_state=1),
           df_features['outbreak_risk'].sample(3000,random_state=1),
           alpha=0.2,s=8,color='#E74C3C')
ax.set_xlabel('Heat-Humidity Stress Index')
ax.set_ylabel('Outbreak Risk Score')
ax.set_title('Heat-Humidity Stress vs Outbreak Risk',fontweight='bold')
plt.tight_layout(); plt.show()

## 5. Train/Test Split

In [ ]:
result = prep.prepare_for_model(df_features)
print(f'Features : {len(result["feature_names"])}')
print(f'Train    : {len(result["X_train"]):,} rows')
print(f'Test     : {len(result["X_test"]):,} rows')
print(f'Target range: {result["y_train"].min():.3f} – {result["y_train"].max():.3f}')

## 6. Summary

| Feature Type | Count | Example |
|---|---|---|
| Climate lags | 15 | precipitation_lag1/2/3 |
| Rolling windows | 6 | precip_roll3, flood_roll6 |
| Cyclical time | 2 | month_sin, month_cos |
| Stress indices | 2 | heat_humid_stress, temp_anomaly |
| Seasonal flags | 1 | wet_season |

➡️ Proceed to **03_Model_Training.ipynb**